<a href="https://colab.research.google.com/github/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/blob/main/telco_bronze_to_silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# telco_bronze_to_silver.ipynb
**Dự án:** Telco Customer Churn – Medallion (Bronze → Silver) với BigQuery + dbt

| Phần | Nội dung | Phụ trách |
|---|---|---|
| 1 | Khảo sát & profiling dataset (21 cột) | Phân tích dataset & Bronze |
| 2 | Nạp CSV vào `bronze_telco.customers_raw` | Phân tích dataset & Bronze |
| 3 | dbt staging / Silver (`stg_telco_customers`, `customers_cleaned`) | Thành viên dbt |

Quy tắc đặt tên: file gốc `telco_customer_churn.csv` · Bronze `bronze_telco.customers_raw` · Silver `silver_telco.customers_cleaned`.

## 0. Cấu hình & xác thực

In [23]:

PROJECT_ID = "btqt2-telco-dbt-bigquery"
BQ_LOCATION = "asia-southeast1"
BRONZE_DATASET = "bronze_telco"
BRONZE_TABLE = "customers_raw"
SOURCE_FILE = "telco_customer_churn.csv"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

print("Project ID:", bq.project)


Project ID: btqt2-telco-dbt-bigquery


## 1. Nạp file CSV gốc
Upload file `WA_Fn-UseC_-Telco-Customer-Churn.csv` (tên gốc từ Kaggle/IBM); ô dưới sẽ đổi tên về `telco_customer_churn.csv` theo quy ước.

In [14]:

import os
import pandas as pd
from urllib.request import urlretrieve

SOURCE_FILE = "telco_customer_churn.csv"

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "huyenpiano1711/BTQT2_Telco_dbt_BigQuery/"
    "main/telco_customer_churn.csv"
)

urlretrieve(DATA_URL, SOURCE_FILE)

df = pd.read_csv(
    SOURCE_FILE,
    dtype=str,
    keep_default_na=False
)

print("Nguồn dữ liệu:", DATA_URL)
print("Kích thước file:", os.path.getsize(SOURCE_FILE), "bytes")
print("Kích thước dataset:", df.shape)

display(df.head())


Nguồn dữ liệu: https://raw.githubusercontent.com/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/main/telco_customer_churn.csv
Kích thước file: 977501 bytes
Kích thước dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


## 2. Khảo sát cấu trúc 21 cột

In [ ]:
import numpy as np
raw = pd.read_csv(SOURCE_FILE)               # đọc tự suy kiểu để thấy kiểu pandas suy ra
summary = pd.DataFrame({
    "kieu_pandas_suy_ra": raw.dtypes.astype(str),
    "so_gia_tri_duy_nhat": df.nunique(),
    "vi_du": df.iloc[0],
})
summary

,kieu_pandas_suy_ra,so_gia_tri_duy_nhat,vi_du
customerID,object,7043,7590-VHVEG
gender,object,2,Female
SeniorCitizen,int64,2,0
Partner,object,2,Yes
Dependents,object,2,No
tenure,int64,73,1
PhoneService,object,2,No
MultipleLines,object,3,No phone service
InternetService,object,3,DSL
OnlineSecurity,object,3,No


In [ ]:
# Phân bố các cột phân loại (<= 6 giá trị)
for c in df.columns:
    if df[c].nunique() <= 6:
        print(f"{c:18s}", df[c].value_counts().to_dict())

gender             {'Male': 3555, 'Female': 3488}
SeniorCitizen      {'0': 5901, '1': 1142}
Partner            {'No': 3641, 'Yes': 3402}
Dependents         {'No': 4933, 'Yes': 2110}
PhoneService       {'Yes': 6361, 'No': 682}
MultipleLines      {'No': 3390, 'Yes': 2971, 'No phone service': 682}
InternetService    {'Fiber optic': 3096, 'DSL': 2421, 'No': 1526}
OnlineSecurity     {'No': 3498, 'Yes': 2019, 'No internet service': 1526}
OnlineBackup       {'No': 3088, 'Yes': 2429, 'No internet service': 1526}
DeviceProtection   {'No': 3095, 'Yes': 2422, 'No internet service': 1526}
TechSupport        {'No': 3473, 'Yes': 2044, 'No internet service': 1526}
StreamingTV        {'No': 2810, 'Yes': 2707, 'No internet service': 1526}
StreamingMovies    {'No': 2785, 'Yes': 2732, 'No internet service': 1526}
Contract           {'Month-to-month': 3875, 'Two year': 1695, 'One year': 1473}
PaperlessBilling   {'Yes': 4171, 'No': 2872}
PaymentMethod      {'Electronic check': 2365, 'Mailed check': 1612, '

In [ ]:
# Thống kê các cột số
num = df[["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]].apply(pd.to_numeric, errors="coerce")
num.describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00,7032.00
mean,0.16,32.37,64.76,2283.30
std,0.37,24.56,30.09,2266.77
min,0.00,0.00,18.25,18.80
25%,0.00,9.00,35.50,401.45
50%,0.00,29.00,70.35,1397.48
75%,0.00,55.00,89.85,3794.74
max,1.00,72.00,118.75,8684.80


## 3. Kiểm tra chất lượng dữ liệu

In [ ]:
# 3.1 Thiếu: NULL thật + chuỗi rỗng/khoảng trắng + token 'NA/null/...'
tokens = {"", "na", "n/a", "null", "nan", "none"}
blank = df.apply(lambda s: s.str.strip().str.lower().isin(tokens)).sum()
print("Cột có giá trị thiếu/trống:")
print(blank[blank > 0] if (blank > 0).any() else "Không có")

# 3.2 Trùng
print("\nDòng trùng hoàn toàn      :", df.duplicated().sum())
print("customerID trùng          :", df.customerID.duplicated().sum())
print("customerID sai định dạng  :", (~df.customerID.str.match(r"^\d{4}-[A-Z]{5}$")).sum())

# 3.3 Khoảng trắng đầu/cuối
ws = df.apply(lambda s: (s != s.str.strip()).sum())
print("\nCột có khoảng trắng thừa:", ws[ws > 0].to_dict())

Cột có giá trị thiếu/trống:
TotalCharges    11
dtype: int64

Dòng trùng hoàn toàn      : 0
customerID trùng          : 0
customerID sai định dạng  : 0

Cột có khoảng trắng thừa: {'TotalCharges': 11}


In [ ]:
# 3.4 Sai kiểu: cột nào không ép được sang số?
for c in ["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]:
    bad = pd.to_numeric(df[c], errors="coerce").isna().sum()
    print(f"{c:15s} không ép được sang số: {bad}")

bad_tc = df[pd.to_numeric(df.TotalCharges, errors="coerce").isna()]
print("\nChi tiết các dòng TotalCharges lỗi:")
display(bad_tc[["customerID","tenure","MonthlyCharges","TotalCharges","Contract","Churn"]]
        .assign(TotalCharges=lambda d: d.TotalCharges.map(repr)))

SeniorCitizen   không ép được sang số: 0
tenure          không ép được sang số: 0
MonthlyCharges  không ép được sang số: 0
TotalCharges    không ép được sang số: 11

Chi tiết các dòng TotalCharges lỗi:


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,' ',Two year,No
753,3115-CZMZD,0,20.25,' ',Two year,No
936,5709-LVOEQ,0,80.85,' ',Two year,No
1082,4367-NUYAO,0,25.75,' ',Two year,No
1340,1371-DWPAZ,0,56.05,' ',Two year,No
3331,7644-OMVMY,0,19.85,' ',Two year,No
3826,3213-VVOLG,0,25.35,' ',Two year,No
4380,2520-SGTTA,0,20,' ',Two year,No
5218,2923-ARZLG,0,19.7,' ',One year,No
6670,4075-WKNIU,0,73.35,' ',Two year,No


In [18]:

# 3.5 Kiểm tra logic nghiệp vụ
import numpy as np
internet_cols = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
ni = df.InternetService == "No"
print("InternetService='No'       :", ni.sum())

for c in internet_cols:
    no_service = df[c] == "No internet service"
    print(f"  {c:17s} 'No internet service' = {no_service.sum()}  | mâu thuẫn = {(ni != no_service).sum()}")

no_phone = df.PhoneService == "No"
no_lines = df.MultipleLines == "No phone service"

print("PhoneService='No'          :", no_phone.sum(),
      "| MultipleLines='No phone service' =", no_lines.sum())
print("Mâu thuẫn dịch vụ điện thoại:", (no_phone != no_lines).sum())

t = pd.to_numeric(df.TotalCharges, errors="coerce")
m = pd.to_numeric(df.MonthlyCharges)
n = pd.to_numeric(df.tenure)

ratio = t / (m * n).replace(0, np.nan)

print("\ntenure = 0                 :", (n == 0).sum())
print("MonthlyCharges <= 0        :", (m <= 0).sum())
print("TotalCharges < MonthlyCharges (tenure>0):", ((t < m) & (n > 0)).sum())
print("TotalCharges / (Monthly*tenure): min={:.2f}, max={:.2f}, std={:.3f}".format(ratio.min(), ratio.max(), ratio.std()))

outlier_mask = (ratio - 1).abs() > 0.5

print("\nSố dòng chênh lệch trên 50%:", outlier_mask.sum())
display(df.loc[outlier_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

print("\nTỷ lệ Churn:", (df.Churn == "Yes").mean().round(4),
      f"({(df.Churn == 'Yes').sum()}/{len(df)})")


InternetService='No'       : 1526
  OnlineSecurity    'No internet service' = 1526  | mâu thuẫn = 0
  OnlineBackup      'No internet service' = 1526  | mâu thuẫn = 0
  DeviceProtection  'No internet service' = 1526  | mâu thuẫn = 0
  TechSupport       'No internet service' = 1526  | mâu thuẫn = 0
  StreamingTV       'No internet service' = 1526  | mâu thuẫn = 0
  StreamingMovies   'No internet service' = 1526  | mâu thuẫn = 0
PhoneService='No'          : 682 | MultipleLines='No phone service' = 682
Mâu thuẫn dịch vụ điện thoại: 0

tenure = 0                 : 11
MonthlyCharges <= 0        : 0
TotalCharges < MonthlyCharges (tenure>0): 0
TotalCharges / (Monthly*tenure): min=0.69, max=1.57, std=0.051

Số dòng chênh lệch trên 50%: 2


,customerID,tenure,MonthlyCharges,TotalCharges
869,0323-XWWTN,3,26.4,121.25
5595,2308-STERM,2,19.4,61.05



Tỷ lệ Churn: 0.2654 (1869/7043)


## 4. Bảng vấn đề dữ liệu

In [20]:

issues = pd.DataFrame([
    ["P1", "TotalCharges", "Sai kiểu / thiếu ngầm",
     "11 dòng chứa khoảng trắng nên không thể chuyển trực tiếp sang kiểu số. Cả 11 dòng đều có tenure=0, Churn=No (10 Two year, 1 One year).",
     11, "Cao",
     "Silver: TRIM và SAFE_CAST sang NUMERIC; giữ NULL đối với giá trị trống."],

    ["P2", "SeniorCitizen", "Quy ước mã hóa",
     "Sử dụng giá trị 0/1 để biểu diễn trạng thái khách hàng cao tuổi, trong khi các cột nhị phân khác dùng Yes/No.",
     7043, "Thấp",
     "Chuyển sang BOOLEAN (is_senior_citizen)."],

    ["P3", "MultipleLines", "Trạng thái nghiệp vụ hợp lệ",
     "682 dòng có giá trị No phone service, tương ứng với PhoneService=No. Không phát hiện mâu thuẫn.",
     682, "Thông tin",
     "Giữ nguyên No phone service để phân biệt với trạng thái No."],

    ["P4", "OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies",
     "Trạng thái nghiệp vụ hợp lệ",
     "1.526 dòng No internet service ở mỗi cột, tương ứng với InternetService=No. Không phát hiện mâu thuẫn.",
     1526, "Thông tin",
     "Giữ nguyên No internet service để bảo toàn ý nghĩa nghiệp vụ."],

    ["P5", "Partner, Dependents, PhoneService, PaperlessBilling, Churn",
     "Kiểu dữ liệu đích",
     "Các cột nhị phân được lưu dưới dạng chuỗi Yes/No.",
     7043, "Thấp",
     "Chuyển sang BOOLEAN trong staging; Churn đổi thành is_churned."],

    ["P6", "tenure, MonthlyCharges, SeniorCitizen",
     "Kiểu dữ liệu thô",
     "Dữ liệu Bronze được lưu dưới dạng STRING; cần chuyển sang kiểu dữ liệu phù hợp trong Silver.",
     7043, "Thấp",
     "Chuyển tenure sang INT64, MonthlyCharges sang NUMERIC và SeniorCitizen sang BOOLEAN."],

    ["P7", "Toàn bộ 21 cột", "Quy ước đặt tên",
     "Tên cột sử dụng nhiều quy ước như customerID, gender, SeniorCitizen và MonthlyCharges.",
     21, "Thấp",
     "Chuẩn hóa tên cột theo snake_case trong dbt staging."],

    ["P8", "TotalCharges vs MonthlyCharges x tenure",
     "Chênh lệch số học cần lưu ý",
     f"Có {int(outlier_mask.sum())} dòng chênh lệch trên 50% so với MonthlyCharges nhân tenure. Chưa đủ căn cứ kết luận dữ liệu sai.",
     int(outlier_mask.sum()), "Thông tin",
     "Giữ nguyên dữ liệu và ghi nhận kết quả kiểm tra; không tự ý điều chỉnh TotalCharges."],

    ["P9", "Churn", "Phân bố lớp (lưu ý)",
     "1.869 khách hàng Churn=Yes (26,54%) và 5.174 khách hàng Churn=No (73,46%).",
     1869, "Thông tin",
     "Giữ nguyên dữ liệu và ghi nhận phân bố lớp."]
], columns=[
    "Mã",
    "Cột",
    "Loại vấn đề",
    "Mô tả",
    "Số dòng ảnh hưởng",
    "Mức độ",
    "Hướng xử lý (Silver)"
])

pd.set_option("display.max_colwidth", 120)
display(issues)


,Mã,Cột,Loại vấn đề,Mô tả,Số dòng ảnh hưởng,Mức độ,Hướng xử lý (Silver)
0,P1,TotalCharges,Sai kiểu / thiếu ngầm,"11 dòng chứa khoảng trắng nên không thể chuyển trực tiếp sang kiểu số. Cả 11 dòng đều có tenure=0, Churn=No (10 Two ...",11,Cao,Silver: TRIM và SAFE_CAST sang NUMERIC; giữ NULL đối với giá trị trống.
1,P2,SeniorCitizen,Quy ước mã hóa,"Sử dụng giá trị 0/1 để biểu diễn trạng thái khách hàng cao tuổi, trong khi các cột nhị phân khác dùng Yes/No.",7043,Thấp,Chuyển sang BOOLEAN (is_senior_citizen).
2,P3,MultipleLines,Trạng thái nghiệp vụ hợp lệ,"682 dòng có giá trị No phone service, tương ứng với PhoneService=No. Không phát hiện mâu thuẫn.",682,Thông tin,Giữ nguyên No phone service để phân biệt với trạng thái No.
3,P4,"OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies",Trạng thái nghiệp vụ hợp lệ,"1.526 dòng No internet service ở mỗi cột, tương ứng với InternetService=No. Không phát hiện mâu thuẫn.",1526,Thông tin,Giữ nguyên No internet service để bảo toàn ý nghĩa nghiệp vụ.
4,P5,"Partner, Dependents, PhoneService, PaperlessBilling, Churn",Kiểu dữ liệu đích,Các cột nhị phân được lưu dưới dạng chuỗi Yes/No.,7043,Thấp,Chuyển sang BOOLEAN trong staging; Churn đổi thành is_churned.
5,P6,"tenure, MonthlyCharges, SeniorCitizen",Kiểu dữ liệu thô,Dữ liệu Bronze được lưu dưới dạng STRING; cần chuyển sang kiểu dữ liệu phù hợp trong Silver.,7043,Thấp,"Chuyển tenure sang INT64, MonthlyCharges sang NUMERIC và SeniorCitizen sang BOOLEAN."
6,P7,Toàn bộ 21 cột,Quy ước đặt tên,"Tên cột sử dụng nhiều quy ước như customerID, gender, SeniorCitizen và MonthlyCharges.",21,Thấp,Chuẩn hóa tên cột theo snake_case trong dbt staging.
7,P8,TotalCharges vs MonthlyCharges x tenure,Chênh lệch số học cần lưu ý,Có 2 dòng chênh lệch trên 50% so với MonthlyCharges nhân tenure. Chưa đủ căn cứ kết luận dữ liệu sai.,2,Thông tin,Giữ nguyên dữ liệu và ghi nhận kết quả kiểm tra; không tự ý điều chỉnh TotalCharges.
8,P9,Churn,Phân bố lớp (lưu ý),"1.869 khách hàng Churn=Yes (26,54%) và 5.174 khách hàng Churn=No (73,46%).",1869,Thông tin,Giữ nguyên dữ liệu và ghi nhận phân bố lớp.


In [21]:
# Xác nhận không có: NULL thật, dòng trùng, customerID trùng
assert df.customerID.is_unique and df.duplicated().sum() == 0
assert len(df) == 7043 and df.shape[1] == 21
print("OK: 7.043 dòng, 21 cột, không trùng.")

OK: 7.043 dòng, 21 cột, không trùng.


## 5. Nạp vào BigQuery Bronze (`bronze_telco.customers_raw`)
Nguyên tắc Bronze: **giữ nguyên dữ liệu gốc** (mọi cột STRING, giữ nguyên `' '` của TotalCharges, giữ nguyên tên cột) + 2 cột metadata để truy vết. Mọi làm sạch/ép kiểu thực hiện ở dbt.

In [24]:
from google.cloud.exceptions import NotFound

dataset_id = f"{PROJECT_ID}.{BRONZE_DATASET}"
ds = bigquery.Dataset(dataset_id); ds.location = BQ_LOCATION
bq.create_dataset(ds, exists_ok=True)

# Schema tường minh: 21 cột STRING + metadata
schema = [bigquery.SchemaField(c, "STRING") for c in df.columns] + [
    bigquery.SchemaField("_ingested_at", "TIMESTAMP"),
    bigquery.SchemaField("_source_file", "STRING"),
]

bronze_df = df.copy()
bronze_df["_ingested_at"] = pd.Timestamp.now(tz="UTC")
bronze_df["_source_file"] = SOURCE_FILE

table_id = f"{dataset_id}.{BRONZE_TABLE}"
job = bq.load_table_from_dataframe(
    bronze_df, table_id,
    job_config=bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE"),  # chạy lại không bị nhân đôi
)
job.result()
print("Đã nạp", bq.get_table(table_id).num_rows, "dòng vào", table_id)

Đã nạp 7043 dòng vào btqt2-telco-dbt-bigquery.bronze_telco.customers_raw


### Kiểm tra sau nạp

In [ ]:
q = f"""
SELECT
  COUNT(*)                                   AS so_dong,
  COUNT(DISTINCT customerID)                 AS so_customer_id,
  COUNTIF(TRIM(TotalCharges) = '')           AS totalcharges_trong,
  COUNTIF(Churn = 'Yes')                     AS so_churn
FROM `{table_id}`
"""
res = bq.query(q).to_dataframe()
display(res)
assert res.so_dong[0] == 7043 and res.so_customer_id[0] == 7043 and res.totalcharges_trong[0] == 11 and res.so_churn[0] == 1869
print("Bronze khớp file nguồn.")
bq.get_table(table_id).schema

## 6. (Tiếp theo) dbt – Staging & Silver
Phần này do thành viên dbt hoàn thiện: project `dbt_telco`, model `stg_telco_customers.sql` → `customers_cleaned.sql` (ghi vào `silver_telco.customers_cleaned`), kiểm thử trong `schema.yml`. Danh sách quy tắc xử lý đã nêu ở bảng vấn đề (mục 4).